<a href="https://colab.research.google.com/github/jessxleee/SIT-UofG-QC-Assignment/blob/main/BB84-Plain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



In [2]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 77.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 7.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=620855c9f61f5004d72731814b2dbd71f1809ff0f5fdffc93f8b81abc8ad6a49
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


Step 1: Setup & Quantum Random Bit Generator

In [3]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

simulator = AerSimulator()

def random_bit():
    """Generate a truly random bit by measuring |+⟩"""
    qc = QuantumCircuit(1, 1) # 1 qubit, 1 classical bit
    qc.h(0)                   # Hadamard: |0⟩ → 1/√2 (|0⟩ + |1⟩)
    qc.measure(0, 0)          # Collapse the superposition
    result = simulator.run(qc, shots=1).result()
    return int(list(result.get_counts().keys())[0])

def random_bits(n):
    return [random_bit() for _ in range(n)]

Step 2 - Alice's Side


In [4]:
N = 100  # number of qubits to send

alice_bits  = random_bits(N)   # the secret bits
alice_bases = random_bits(N)   # 0 = rectilinear (+), 1 = diagonal (×)

def encode_qubit(bit, basis):
    """Alice encodes one qubit"""
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)          # flip to |1⟩
    if basis == 1:
        qc.h(0)          # rotate to diagonal basis
    return qc

Step 3 - Bob's Side

In [5]:
bob_bases = random_bits(N)

def measure_qubit(qc, basis):
    """Bob measures in his chosen basis"""
    if basis == 1:
        qc.h(0)          # rotate back from diagonal
    qc.measure(0, 0)
    result = simulator.run(qc, shots=1).result()
    return int(list(result.get_counts().keys())[0])

bob_results = []
for i in range(N):
    qc = encode_qubit(alice_bits[i], alice_bases[i])
    bit = measure_qubit(qc, bob_bases[i])
    bob_results.append(bit)

Step 4 — Sifting (Classical Channel)

In [6]:
# === SIFTING ===
sifted_alice = []
sifted_bob   = []

for i in range(N):
    if alice_bases[i] == bob_bases[i]:   # bases matched
        sifted_alice.append(alice_bits[i])
        sifted_bob.append(bob_results[i])

print(f"Sifted key length: {len(sifted_alice)} bits")

Sifted key length: 48 bits


Step 5 - Error Checking

In [7]:
sample_size = 10  # bits to sacrifice for checking
THRESHOLD   = 0.1

sample_alice = sifted_alice[:sample_size]
sample_bob   = sifted_bob[:sample_size]

errors = sum(a != b for a, b in zip(sample_alice, sample_bob))
error_rate = errors / sample_size

print("=" * 45)
print("       BB84 PROTOCOL — SECURITY REPORT")
print("=" * 45)
print(f"  Qubits sent:          {N}")
print(f"  Sample checked:       {sample_size} bits")
print(f"  Errors found:         {errors}/{sample_size}")
print(f"  Error rate:           {error_rate:.0%}")
print(f"  Detection threshold:  {THRESHOLD:.0%}")
print("-" * 45)

if error_rate > THRESHOLD:
    print("Key exchange ABORTED. Do not use key.")
else:
    final_key = sifted_alice[sample_size:]  # remaining bits = shared key
    print(f" No attack detected. Key length: {len(final_key)} bits")
    print(f"Key: {final_key}")

       BB84 PROTOCOL — SECURITY REPORT
  Qubits sent:          100
  Sifted key length:    48 bits
  Sample checked:       10 bits
  Errors found:         0/10
  Error rate:           0%
  Detection threshold:  10%
---------------------------------------------
 No attack detected. Key length: 38 bits
Key: [1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0]
